# Property assessment ontology and data agent lab

Create `PropertyAssessmentOntology` over the managed Gold marts, validate its business graph, and ground a Fabric data agent in that ontology. Complete these steps in the Fabric portal and record your evidence in new markdown cells.

## 1. Confirm prerequisites

Confirm that notebook 4 completed, all four tables are present in `GoldLakehouse.dbo`, and **Ontology item (preview)** is enabled for the tenant. Also confirm that the separate Fabric data agent and required Copilot tenant settings are enabled. The bound Gold tables must be managed Delta tables without OneLake security or Delta column mapping enabled.

## 2. Create the ontology

Create an Ontology item named `PropertyAssessmentOntology` and choose to build directly from OneLake. Use the Goal 4 ontology guide in the repository Documents folder for the exact property mappings.

| Entity type | Entity key | Display property | Gold source |
|---|---|---|---|
| `Property` | `PropertyId` from `parcel_id` | `PropertyNumber` | `property_assessment_mart` |
| `Neighborhood` | `NeighborhoodId` from `neighborhood_id` | `NeighborhoodName` | `neighborhood_equity_mart` |
| `ComparableSale` | `SaleId` from `sale_id` | `SaleDate` | `comparable_sales_mart` |
| `Appeal` | `AppealId` from `appeal_id` | `AppealId` | `appeal_intelligence_mart` |

Do not bind the original appeal `narrative`. Bind the governed AI summary, classification, sentiment, and follow-up properties instead.

## 3. Bind relationships

| Relationship | Mapping table | Origin match | Target match |
|---|---|---|---|
| `Property` -- `locatedIn` --> `Neighborhood` | `property_assessment_mart` | `parcel_id` | `neighborhood_id` |
| `ComparableSale` -- `recordsFor` --> `Property` | `comparable_sales_mart` | `sale_id` | `parcel_id` |
| `Appeal` -- `submittedFor` --> `Property` | `appeal_intelligence_mart` | `appeal_id` | `parcel_id` |
| `Appeal` -- `occursIn` --> `Neighborhood` | `appeal_intelligence_mart` | `appeal_id` | `neighborhood_id` |

Save each relationship and confirm that all four appear on the ontology canvas.

## 4. Refresh and validate the graph

Refresh the graph model after saving the bindings. Inspect entity instances and verify that Property traverses to one Neighborhood, ComparableSale traverses to one Property, and Appeal traverses to both Property and Neighborhood. Confirm that numeric values are populated. If a required decimal property appears null in the preview experience, retain the authoritative decimal in Gold and bind a `double`-typed ontology projection instead.

## 5. Define the agent's job and source

Create `Property Assessment Agent`. Attach `PropertyAssessmentOntology` as its only source for this exercise so the evaluation proves that entity and relationship grounding works. The agent answers questions about synthetic property values, comparable sales, inspections, appeal operations, and AI-derived appeal themes. Explicitly exclude real valuation advice, legal conclusions, and automated appeal decisions.

## 6. Add business instructions

Add `Support group by in GQL`, which is the current Microsoft-documented preview guidance for ontology aggregations. Tell the agent that assessed value, sale price, and estimated tax are different measures. Define assessment-to-sale ratio, appeal rate, negative-sentiment rate, and inspection follow-up. Require the agent to state that all values are synthetic and to use ontology relationships rather than inventing joins.

## 7. Add safety instructions

- Never claim that an assessment or tax calculation is legally authoritative.
- Never recommend approving or rejecting an appeal from sentiment.
- Distinguish source fields from AI-derived fields.
- Do not quote or reconstruct an original appeal narrative.
- State when the ontology cannot answer a question.
- Ask for clarification when tax year, neighbourhood, or property class is ambiguous.

## 8. Evaluate relationship-aware questions

Test these questions and inspect the generated grounding or graph query:

1. Which properties have both a comparable sale and an appeal?
2. Show appeals for properties whose assessment-to-sale ratio is above 1.1.
3. Which neighbourhood has the highest negative appeal sentiment rate?
4. For a selected parcel, show its neighbourhood, comparable sales, and appeals.
5. Which appealed properties also require inspection follow-up?
6. Explain why an exact tax bill cannot be treated as legally authoritative in this dataset.

Compare numeric answers with the Gold tables. Refine entity descriptions, relationship bindings, or agent instructions whenever the query uses the wrong grain, measure, or path.

## Completion evidence

Record the ontology canvas, entity keys, four relationship bindings, graph refresh status, one successful entity traversal, final agent instructions, evaluation results, and one example where the agent correctly declined or qualified an unsupported request.